In [75]:
#step3:load the interview Guide with LangChain

with open("interview_guide.txt", "w", encoding="utf-8") as f:
    f.write("""AI Engineer Interview Guide

Python:
Important topics include variables, functions, OOP, lists, dictionaries, exception handling, and file handling.

Machine Learning:
Important topics include supervised learning, unsupervised learning, regression, classification, overfitting, underfitting, and evaluation metrics.

Deep Learning:
Important topics include neural networks, activation functions, backpropagation, CNNs, RNNs, LSTMs, and Transformers.

NLP:
Important topics include tokenization, embeddings, attention, sequence models, and text classification.

Generative AI:
Important topics include LLMs, prompting, fine-tuning, RAG, embeddings, vector databases, and agents.

LangChain:
Important topics include prompt templates, chains, output parsers, document loaders, text splitters, retrievers, and vector stores.
""")

print("Interview guide written successfully!")

Interview guide written successfully!


In [76]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader(
    "interview_guide.txt",
    encoding="utf-8"
)

documents = loader.load()
print(documents[0].page_content)

AI Engineer Interview Guide

Python:
Important topics include variables, functions, OOP, lists, dictionaries, exception handling, and file handling.

Machine Learning:
Important topics include supervised learning, unsupervised learning, regression, classification, overfitting, underfitting, and evaluation metrics.

Deep Learning:
Important topics include neural networks, activation functions, backpropagation, CNNs, RNNs, LSTMs, and Transformers.

NLP:
Important topics include tokenization, embeddings, attention, sequence models, and text classification.

Generative AI:
Important topics include LLMs, prompting, fine-tuning, RAG, embeddings, vector databases, and agents.

LangChain:
Important topics include prompt templates, chains, output parsers, document loaders, text splitters, retrievers, and vector stores.



In [77]:
#step4:split the interview guide

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}")
    print(chunk.page_content)

Number of chunks: 4

Chunk 1
AI Engineer Interview Guide

Python:
Important topics include variables, functions, OOP, lists, dictionaries, exception handling, and file handling.

Chunk 2
Machine Learning:
Important topics include supervised learning, unsupervised learning, regression, classification, overfitting, underfitting, and evaluation metrics.

Chunk 3
Deep Learning:
Important topics include neural networks, activation functions, backpropagation, CNNs, RNNs, LSTMs, and Transformers.

NLP:
Important topics include tokenization, embeddings, attention, sequence models, and text classification.

Chunk 4
Generative AI:
Important topics include LLMs, prompting, fine-tuning, RAG, embeddings, vector databases, and agents.

LangChain:
Important topics include prompt templates, chains, output parsers, document loaders, text splitters, retrievers, and vector stores.


In [78]:
#step5:create embeddings 
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embeddings loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 925.06it/s]


Embeddings loaded successfully!


In [79]:
#step6:convert the interview chunks into embeddings

chunk_vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]

)

print("Number of vectors:", len(chunk_vectors))
print("Vector size:", len(chunk_vectors[0]))

Number of vectors: 4
Vector size: 384


In [80]:
#step7:create the FAISS vector

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Vector store loaded successfully!")

Vector store loaded successfully!


In [81]:
#step8:create the retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

docs = retriever.invoke("What should I study for a Python interview?")

for doc in docs:
    print(doc.page_content)
    print("-" * 40)

AI Engineer Interview Guide

Python:
Important topics include variables, functions, OOP, lists, dictionaries, exception handling, and file handling.
----------------------------------------
Machine Learning:
Important topics include supervised learning, unsupervised learning, regression, classification, overfitting, underfitting, and evaluation metrics.
----------------------------------------
Generative AI:
Important topics include LLMs, prompting, fine-tuning, RAG, embeddings, vector databases, and agents.

LangChain:
Important topics include prompt templates, chains, output parsers, document loaders, text splitters, retrievers, and vector stores.
----------------------------------------


In [ ]:
#step9:connect the retriever to the llm
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    max_new_tokens=150 ,
    device=-1,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)
chat_model = ChatHuggingFace(llm=llm)
print("Chat model loaded!")

Loading weights: 100%|██████████| 434/434 [00:17<00:00, 25.21it/s] 


Chat model loaded!


In [83]:
#step10:Build the RAG Chain(Connect Retriever → Prompt → LLM)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_prompt = ChatPromptTemplate.from_template(
    """You are an AI Interview Assistant.
Answer the question ONLY using the information in the Context.

If the answer is not present in the Context, say:
"I don't have this information in the interview guide."

Do not use your own knowledge.

Context:
{context}

Question:
{question}

Answer briefly.
"""
)

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | chat_model
    | StrOutputParser()
)

response = rag_chain.invoke(
    "What topics should I study for a Python interview?"
)

print(response)

""""context": retriever

means:

Take the user's question → send it to the retriever → put the retrieved documents into {context}.
"question": RunnablePassthrough()

means:

Pass the original question directly into {question}."""

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Python:
- Variables
- Functions
- OOP
- Lists
- Dictionaries
- Exception handling
- File handling


'"context": retriever\n\nmeans:\n\nTake the user\'s question → send it to the retriever → put the retrieved documents into {context}.\n"question": RunnablePassthrough()\n\nmeans:\n\nPass the original question directly into {question}.'

In [84]:
#step11:test another topic
response = rag_chain.invoke(
    "What topics should I study for a Generative AI interview?"
)
print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LLM, prompting, fine-tuning, RAG, embeddings, vector databases, agents.


In [85]:
#step12:test an unrelated question

response = rag_chain.invoke(
    "What is the capital of India?"
)
print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I don't have this information in the interview guide.


In [86]:
#step13:Add Resume PDF Support
!pip install pypdf



In [87]:
#step14:load the resume pdf
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("resume.pdf")
resume_documents = loader.load()

print("Number of pages:", len(resume_documents))
print(resume_documents[0].page_content[:1000])

"""This will load the resume and print the first 1000 characters."""

Number of pages: 1
AI Engineer Resume
Career Objective
Aspiring AI Engineer with hands-on experience in Python, PyTorch, Deep Learning, NLP,
Generative AI, RAG, LangChain, and LLM applications.
Technical Skills
Python, NumPy, Pandas, PyTorch, Deep Learning, CNN, RNN, LSTM, GRU, Transformers, NLP,
Generative AI, RAG, Embeddings, FAISS, LangChain, LangGraph, LLM Fine-tuning.
Projects
AI Interview Preparation Assistant — Built a RAG-based assistant using LangChain, Hugging Face
models, embeddings, FAISS, document retrieval, and an interview knowledge base.
Customer Support AI Assistant — Built a text classification assistant using DistilBERT and LoRA.
AI Research Assistant — Built a PDF RAG system using Sentence Transformers, FAISS,
reranking, FLAN-T5, source-page retrieval, and conversation memory.
Education
B.Tech in Artificial Intelligence, Lords Institute of Engineering and Technology.


'This will load the resume and print the first 1000 characters.'

In [88]:
#step15:split the resume into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

resume_chunks = splitter.split_documents(resume_documents)

print("Number of resume chunks:", len(resume_chunks))

for i, chunk in enumerate(resume_chunks):
    print(f"\nChunk {i+1}:")
    print(chunk.page_content)



Number of resume chunks: 4

Chunk 1:
AI Engineer Resume
Career Objective
Aspiring AI Engineer with hands-on experience in Python, PyTorch, Deep Learning, NLP,
Generative AI, RAG, LangChain, and LLM applications.
Technical Skills
Python, NumPy, Pandas, PyTorch, Deep Learning, CNN, RNN, LSTM, GRU, Transformers, NLP,

Chunk 2:
Generative AI, RAG, Embeddings, FAISS, LangChain, LangGraph, LLM Fine-tuning.
Projects
AI Interview Preparation Assistant — Built a RAG-based assistant using LangChain, Hugging Face
models, embeddings, FAISS, document retrieval, and an interview knowledge base.

Chunk 3:
Customer Support AI Assistant — Built a text classification assistant using DistilBERT and LoRA.
AI Research Assistant — Built a PDF RAG system using Sentence Transformers, FAISS,
reranking, FLAN-T5, source-page retrieval, and conversation memory.
Education

Chunk 4:
Education
B.Tech in Artificial Intelligence, Lords Institute of Engineering and Technology.


In [89]:
#step16:create resume embeddings
#we'll convert those 4 chunks into vectors so our system can search the resume semantically.


resume_vectorstore = FAISS.from_documents(
    documents=resume_chunks,
    embedding=embeddings

)

print("Resume vector store created successfully!")

Resume vector store created successfully!


In [90]:
#step18:create a resume retriever
resume_retriever = resume_vectorstore.as_retriever(
    search_kwargs={"k": 2}

)

docs = resume_retriever.invoke(
    "What Python and AI skills does the candidate have?"
)

for doc in docs:
    print(doc.page_content)
    print("-" * 40)

AI Engineer Resume
Career Objective
Aspiring AI Engineer with hands-on experience in Python, PyTorch, Deep Learning, NLP,
Generative AI, RAG, LangChain, and LLM applications.
Technical Skills
Python, NumPy, Pandas, PyTorch, Deep Learning, CNN, RNN, LSTM, GRU, Transformers, NLP,
----------------------------------------
Customer Support AI Assistant — Built a text classification assistant using DistilBERT and LoRA.
AI Research Assistant — Built a PDF RAG system using Sentence Transformers, FAISS,
reranking, FLAN-T5, source-page retrieval, and conversation memory.
Education
----------------------------------------


In [91]:
#step18:connect resume retirever to the llm

resume_prompt = ChatPromptTemplate.from_template(
    """You are an AI Resume Assistant.

Answer the question ONLY using the information in the context.

If the answer is not present in the Context, say:
"I don't have this information in the resume."  

Do not use your own knowledge.

Context:
{context}

Question:
{question}

Answer briefly.
"""
)


resume_chain = (
    {
        "context": resume_retriever,
        "question": RunnablePassthrough()

    }

    | resume_prompt
    | chat_model
    | StrOutputParser()
)

response = resume_chain.invoke(
    "What deep learning skiils does the candidate have?"
)
print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The candidate has skills in CNN, RNN, LSTM, GRU, and Transformers.


In [92]:
#step19:test a missing skill

response = resume_chain.invoke(
    "Does the candidate have experience with Java?"
)
print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I don't have this information in the resume.


In [93]:
#step20:compare resume skills with interview requirements
#We'll give the system:

"""The resume
The interview guide
A target area, such as Generative AI

Then it can identify what the candidate has and what they should study."""

comparison_prompt = ChatPromptTemplate.from_template(
    """You are an AI Interview Preparation Assistant.

Compare the candidate's resume with the interview guide.

Resume:
{resume}

Interview Guide:
{guide}

For the requested topic, identify:
1. Skills/topics already present in the resume.
2. Skills/topics mentioned in the interview guide but not clearly present in the resume.

Do not invent information.

Topic: {topic}

Answer briefly.
"""
)


In [94]:
#step21:connect  both retrievers

"""resume_retriever → candidate's skills
retriever → interview requirements"""

comparison_chain = (
    {
        "resume": resume_retriever,
        "guide": retriever,
        "topic": RunnablePassthrough()

    }

    | comparison_prompt
    | chat_model
    | StrOutputParser()

)

response = comparison_chain.invoke(
    "Generative AI"
)

print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Skills/Topics Already Present in the Resume:
1. **Generative AI**
2. **LLMs**
3. **Prompting**
4. **Fine-tuning**
5. **RAG**
6. **Embeddings**
7. **Vector databases**

### Skills/Topics Mentioned in the Interview Guide but Not Clearly Present in the Resume:
1. **Agents**


In [95]:
#step22:test python skill gap
response = comparison_chain.invoke(
    "Python"
)
print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Skills/topics already present in the resume for Python:
- Python
- NumPy
- Pandas

Skills/topics mentioned in the interview guide but not clearly present in the resume:
- Variables
- Functions
- OOP
- Lists
- Dictionaries
- Exception handling
- File handling


In [96]:
#step24:make the output more structured
comparison_prompt = ChatPromptTemplate.from_template(
    """You are an AI Interview Preparation Assistant.

Compare the candidate's resume with the interview guide.

Resume:
{resume}

Interview Guide:
{guide}

Topic: {topic}

Return exactly these three sections:

Skills Already Present:
- List skills clearly supported by the resume.

Topics to Study:
- List topics from the interview guide that are not clearly supported by the resume.

Interview Preparation:
- Give one short preparation suggestion based only on the information above.

Do not invent information.
"""
)

In [97]:
comparison_chain = (
    {
        "resume": resume_retriever,
        "guide": retriever,
        "topic": RunnablePassthrough()

    }

    | comparison_prompt
    | chat_model
    | StrOutputParser()

)

response = comparison_chain.invoke(
    "Generative AI"
)

print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Skills Already Present:
- Generative AI
- RAG
- Embeddings
- FAISS
- LangChain
- LangGraph
- LLM Fine-tuning

Topics to Study:
- Agents
- Prompting
- Vector Databases

Interview Preparation:
- Practice crafting effective prompts for LLMs to ensure better interaction and results.


In [98]:
response = comparison_chain.invoke(
    "Python"
)
print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Skills Already Present:
- Python
- NumPy
- Pandas
- PyTorch
- Deep Learning
- CNN
- RNN
- LSTM
- GRU
- Transformers
- NLP

Topics to Study:
- Exception handling
- File handling

Interview Preparation:
- Practice implementing basic Python concepts such as variables, functions, and exception handling, as these are important but not explicitly mentioned in your resume.


In [99]:
#step25:build the mock interview question generator
question_prompt = ChatPromptTemplate.from_template(
    """You are an AI Interviewer.

Generate exactly 5 simple interview questions.

Topic: {topic}

Allowed topics ONLY:
Variables, Functions, OOP, Lists, Dictionaries,
Exception Handling, File Handling.

Candidate Resume:
{resume}

Interview Guide:
{guide}

Rules:
- Each question must be about one of the allowed topics.
- Do NOT ask about any subtopic not explicitly listed above.
- Prefer topics that are not clearly present in the resume.
- Do not provide answers.
- Generate exactly 5 questions.
"""



        

)

print("Question prompt created!")

Question prompt created!


In [100]:
#step26:connect both retrievers to the question
question_chain = (
    {
        "resume": resume_retriever,
        "guide": retriever,
        "topic": RunnablePassthrough()
    }
    | question_prompt
    | chat_model
    | StrOutputParser()

)

response = question_chain.invoke("Python")

print(response)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. Can you explain how to define and use variables in Python, including the difference between local and global variables?

2. How would you implement a function in Python that takes a list of integers as input and returns the sum of all even numbers in the list?

3. Could you demonstrate your understanding of object-oriented programming (OOP) in Python by creating a class for a book with attributes like title, author, and publication year, and methods to update these attributes?

4. Please write a Python script that demonstrates the usage of lists, including adding elements, removing elements, and iterating through the list.

5. How would you handle exceptions when working with file handling in Python? Please provide an example of reading from a file using try-except


In [105]:
#step27:mock interview(we'll make the assistant ask one question at a time and evaluate the candidate's answer.)

evaluation_prompt = ChatPromptTemplate.from_template(
    """You are an AI Interview Evaluator.

Evaluate the candidate's answer.

Question:
{question}

Candidate Answer:
{answer}

Return ONLY this format:

Score: <number from 1 to 10>
Strengths: <one short sentence>
Improvement: <one short sentence>
Ideal Answer: <one short sentence>

Do not add anything else.
"""

)
print("Evaluation prompt created!")

Evaluation prompt created!


In [106]:
#step28:Connect the Evaluation Prompt to the LLM

evaluation_chain = (
    evaluation_prompt
    | chat_model
    | StrOutputParser()
)

print("Evaluation chain created!")

Evaluation chain created!


In [107]:
#step29:test the evaluation chain

evaluation = evaluation_chain.invoke({
    "question": "What is a variable in Python?",
    "answer": "A variable is a name used to store a value."


})

print(evaluation)


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: 3
Strengths: The answer provides a basic definition of a variable.
Improvement: It should include that variables can hold different data types and are essential for storing data in Python.
Ideal Answer: A variable in Python is a named reference to a value, which can be of various data types like integers, strings, or lists.


In [109]:
#step30:create the mock interview loop

evaluations = []
questions = [
    "What is variable in Python?",
    "What is a function in Python?",
    "What is Object-Oriented Programming (OOP)?"
]

for i, question in enumerate(questions, 1):
    print(f"\nQuestion {i}: {question}")

    answer = input("Your answer: ")

    evaluation = evaluation_chain.invoke({
        "question": question,
        "answer": answer
    })

    print("\nEvaluation:")
    print(evaluation)

    evaluations.append(evaluation)


Question 1: What is variable in Python?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Evaluation:
Score: 4
Strengths: The candidate provided a clear understanding of what variables represent in Python.
Improvement: A more accurate description would be needed, such as "symbolic names that hold references to values."
Ideal Answer: A symbolic name that holds a reference to an object.

Question 2: What is a function in Python?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Evaluation:
Score: 7
Strengths: The candidate provided a correct definition of a function.
Improvement: It would be more complete to mention that functions can take parameters and return values.
Ideal Answer: A function in Python is a reusable block of code that performs a specific task and can accept parameters and return results.

Question 3: What is Object-Oriented Programming (OOP)?


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Evaluation:
Score: 7
Strengths: The candidate provided a basic understanding of OOP.
Improvement: The answer could be more detailed by explaining key concepts like encapsulation, inheritance, and polymorphism.
Ideal Answer: Object-Oriented Programming (OOP) is a programming paradigm that organizes code into objects, which are instances of classes that encapsulate data and behavior. It emphasizes modularity, reusability, and abstraction.


In [ ]:
#step31:calculate average score

import re

scores = []

for evaluation in evaluations:
    match = re.search(r"Score:\s*(\d+)", evaluation)
    if match:
        scores.append(int(match.group(1)))

if scores:
    average_score = sum(scores) / len(scores)
    print(f"Average Interview Score: {average_score:.1f}/10")

    """1. import re

re is Python's Regular Expression module.

We use it to find the score inside text like:

Score: 7

This searches for something like:

Score: 7

The pattern:

Score: → looks for the word Score:
\s* → allows spaces
\d+ → finds one or more digits
match.group(1) gets the number found.

For example:

match.group(1)

gives:

"7"

int() converts it from text to a number:

int("7") → 7"""

Average Interview Score: 6.0/10


In [117]:
#step32:add skill-gap summary
report_prompt = ChatPromptTemplate.from_template(
    """You are an AI Interview Preparation Assistant.

Create a concise interview performance report.

Average Score: {score}
Skill Gap: {skill_gap}
Evaluations: {evaluations}

Return exactly these sections:

Overall Score:
Skill Strengths:
Topics to Improve:
Interview Performance:

For Interview Performance, give ONE short sentence summarizing the candidate's performance.
Do not include ideal answers or detailed explanations.
"""
)

print("Report prompt updated!")

Report prompt updated!


In [119]:
#step33:connect the report to your data

skill_gap = comparison_chain.invoke("Python")

evaluation_text = "\n\n".join(evaluations)

final_report = report_prompt | chat_model | StrOutputParser()

report = final_report.invoke({
    "score": average_score,
    "skill_gap": skill_gap,
    "evaluations": evaluation_text
})
print(report)

"""we’ll actually generate the final interview performance report using:

average_score
evaluations
Skill-gap analysis from your comparison_chain.

What this does
skill_gap → gets your Python skill-gap analysis.
evaluation_text → combines all interview evaluations into one string.
final_report → creates the report chain.
report → generates your final performance report."""

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Overall Score: 6.0
Skill Strengths: The candidate demonstrated a strong grasp of foundational skills including Python, NumPy, Pandas, PyTorch, deep learning, and various neural network models. They also showed proficiency in Natural Language Processing (NLP).
Topics to Improve: The candidate needs to enhance their understanding of exception handling and file handling concepts.
Interview Performance: The candidate performed moderately well but lacked depth in explaining some fundamental concepts.


'we’ll actually generate the final interview performance report using:\n\naverage_score\nevaluations\nSkill-gap analysis from your comparison_chain.\n\nWhat this does\nskill_gap → gets your Python skill-gap analysis.\nevaluation_text → combines all interview evaluations into one string.\nfinal_report → creates the report chain.\nreport → generates your final performance report.'

In [122]:


final_report = report_prompt | chat_model | StrOutputParser()

report = final_report.invoke({
    "score": average_score,
    "skill_gap": skill_gap,
    "evaluations": evaluation_text
})

print(report) 

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Overall Score:
6.0

Skill Strengths:
- Python
- NumPy
- Pandas
- PyTorch
- Deep Learning
- CNN
- RNN
- LSTM
- GRU
- Transformers
- NLP

Topics to Improve:
- Exception handling
- File handling

Interview Performance:

The candidate demonstrated a solid grasp of fundamental Python concepts but showed room for improvement in specific areas such as exception handling and file handling, indicating a need for further practice and deeper understanding in these topics.


In [123]:
#step34:save the final report
with open("interview_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

print("Interview report saved successfully!")

Interview report saved successfully!


In [124]:
#step35:add a simple project summary
project_summary = """AI Interview Preparation Assistant

This project helps candidates prepare for AI Engineer interviews.

Main Features:
- Resume-based skill analysis
- Interview guide retrieval using RAG
- Skill-gap analysis
- Interview question generation
- Candidate answer evaluation
- Interview score calculation
- Final performance report generation

Technologies:
- Python
- PyTorch
- Hugging Face Transformers
- LangChain
- FAISS
- Sentence Transformers
- Qwen2.5-3B-Instruct
"""

with open("project_summary.txt", "w", encoding="utf-8") as f:
    f.write(project_summary)

print("Project summary saved successfully!")

Project summary saved successfully!


In [ ]:
#step36:create requirements.txt(This file lists the Python libraries needed to run your project.)


requirements = """langchain
langchain-core
langchain-community
langchain-huggingface
langchain-text-splitters
transformers==4.57.3
torch
sentence-transformers
faiss-cpu
pypdf
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

print("requirements.txt created successfully!")

"""Purpose: Anyone can install all project dependencies later using:

pip install -r requirements.txt"""

requirements.txt created successfully!


In [126]:
#step37:check your project files

import os
print(os.listdir())

['interview_assistant,ipynb.ipynb', 'interview_guide.txt', 'interview_report.txt', 'langchain', 'project_summary.txt', 'requirements.txt', 'resume.pdf']


FileNotFoundError: [WinError 2] The system cannot find the file specified: 'interview_assistant,ipynb.ipynb' -> 'interview_assistant.ipynb'